In [5]:
"""
Evaluation script for trained binary ROI classifiers.

Runs on the held-out test set and reports:
  - Accuracy
  - Precision, Recall, F1 (per-class and macro)
  - Confusion matrix
  - ROC AUC
  - Saves a confusion matrix plot
"""

from __future__ import annotations

import sys
import os
from pathlib import Path

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from torch.utils.data import DataLoader
from tqdm import tqdm

project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.dataset.eyes_dataset import EyesDataset, _VAL_TRANSFORMS
from src.models.binary_roi_classifier import build_binary_classifier


# =========================
# Config
# =========================
TEST_ROOTS = {
    "eyes":  "../dataset_split/test/eyes",
    "mouth": "../dataset_split/test/mouth",
}

CLASS_LABELS = {
    "eyes":  {"closed": 0, "open": 1},
    "mouth": {"closed": 0, "open": 1},
}

CHECKPOINT_DIR = Path("runs/binary")
RESULTS_DIR    = Path("runs/eval")

BATCH_SIZE   = 64
NUM_WORKERS  = 4


# =========================
# Helpers
# =========================
def load_test_loader(roi: str) -> DataLoader:
    root = Path(TEST_ROOTS[roi])
    dataset = EyesDataset(root, CLASS_LABELS[roi], transform=_VAL_TRANSFORMS)
    print(f"  Test samples : {len(dataset)}")

    label_counts = {}
    for _, label in dataset.samples:
        label_counts[label] = label_counts.get(label, 0) + 1
    for label, count in sorted(label_counts.items()):
        class_name = [k for k, v in CLASS_LABELS[roi].items() if v == label][0]
        print(f"    {class_name:<10}: {count}")

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )


def run_inference(model, loader, device) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Returns (all_labels, all_preds, all_probs)."""
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating"):
            images = images.to(device)
            logits = model(images)
            probs  = torch.softmax(logits, dim=1)

            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_probs.extend(probs[:, 1].cpu().tolist())   # prob of positive class
            all_labels.extend(labels.tolist())

    return (
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_probs),
    )


# =========================
# Plots
# =========================
def plot_confusion_matrix(
    cm: np.ndarray,
    class_names: list[str],
    roi: str,
    save_path: Path,
) -> None:
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"Confusion Matrix — {roi.upper()}")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  Saved confusion matrix -> {save_path}")


def plot_roc_curve(
    labels: np.ndarray,
    probs: np.ndarray,
    roi: str,
    save_path: Path,
) -> None:
    fpr, tpr, _ = roc_curve(labels, probs)
    auc = roc_auc_score(labels, probs)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, label=f"AUC = {auc:.3f}", color="steelblue", lw=2)
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curve — {roi.upper()}")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  Saved ROC curve        -> {save_path}")


# =========================
# Evaluate
# =========================
def evaluate_roi(roi: str) -> None:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*50}")
    print(f"Evaluating ROI : {roi.upper()} on {device}")
    print(f"{'='*50}")

    # --- Paths ---
    checkpoint_path = CHECKPOINT_DIR / roi / "best_1.pt"
    if not checkpoint_path.exists():
        print(f"  ERROR: No checkpoint found at {checkpoint_path}")
        return

    results_dir = RESULTS_DIR / roi
    results_dir.mkdir(parents=True, exist_ok=True)

    # --- Data ---
    print("\nTest set:")
    loader = load_test_loader(roi)

    # --- Model ---
    model = build_binary_classifier(pretrained=False, freeze_backbone=False).to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"\n  Loaded checkpoint: {checkpoint_path}")

    # --- Inference ---
    labels, preds, probs = run_inference(model, loader, device)

    # --- Metrics ---
    class_names = [k for k, v in sorted(CLASS_LABELS[roi].items(), key=lambda x: x[1])]
    acc = accuracy_score(labels, preds)
    auc = roc_auc_score(labels, probs)
    cm  = confusion_matrix(labels, preds)

    print(f"\n--- Results ---")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  ROC AUC  : {auc:.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(labels, preds, target_names=class_names, zero_division=0))
    print(f"  Confusion Matrix:")
    print(f"    Labels: {class_names}")
    print(f"    {cm}")

    # --- Save results to txt ---
    report_path = results_dir / "report.txt"
    with open(report_path, "w") as f:
        f.write(f"ROI: {roi.upper()}\n")
        f.write(f"Checkpoint: {checkpoint_path}\n\n")
        f.write(f"Accuracy : {acc:.4f}\n")
        f.write(f"ROC AUC  : {auc:.4f}\n\n")
        f.write("Classification Report:\n")
        f.write(classification_report(labels, preds, target_names=class_names, zero_division=0))
        f.write(f"\nConfusion Matrix ({class_names}):\n")
        f.write(str(cm))
    print(f"\n  Saved report           -> {report_path}")

    # --- Plots ---
    plot_confusion_matrix(cm, class_names, roi, results_dir / "confusion_matrix.png")
    plot_roc_curve(labels, probs, roi, results_dir / "roc_curve.png")


# =========================
# Run
# =========================
if __name__ == "__main__":
    evaluate_roi("eyes")
    evaluate_roi("mouth")


Evaluating ROI : EYES on cuda

Test set:
  Test samples : 4118
    closed    : 405
    open      : 3713

  Loaded checkpoint: runs\binary\eyes\best_1.pt


Evaluating: 100%|██████████| 65/65 [00:11<00:00,  5.54it/s]



--- Results ---
  Accuracy : 0.9582
  ROC AUC  : 0.9918

  Classification Report:
              precision    recall  f1-score   support

      closed       0.72      0.96      0.82       405
        open       0.99      0.96      0.98      3713

    accuracy                           0.96      4118
   macro avg       0.86      0.96      0.90      4118
weighted avg       0.97      0.96      0.96      4118

  Confusion Matrix:
    Labels: ['closed', 'open']
    [[ 387   18]
 [ 154 3559]]

  Saved report           -> runs\eval\eyes\report.txt
  Saved confusion matrix -> runs\eval\eyes\confusion_matrix.png
  Saved ROC curve        -> runs\eval\eyes\roc_curve.png

Evaluating ROI : MOUTH on cuda

Test set:
  Test samples : 2085
    closed    : 2048
    open      : 37

  Loaded checkpoint: runs\binary\mouth\best_1.pt


Evaluating: 100%|██████████| 33/33 [00:13<00:00,  2.52it/s]


--- Results ---
  Accuracy : 0.9353
  ROC AUC  : 0.9917

  Classification Report:
              precision    recall  f1-score   support

      closed       1.00      0.93      0.97      2048
        open       0.21      0.97      0.35        37

    accuracy                           0.94      2085
   macro avg       0.61      0.95      0.66      2085
weighted avg       0.99      0.94      0.95      2085

  Confusion Matrix:
    Labels: ['closed', 'open']
    [[1914  134]
 [   1   36]]

  Saved report           -> runs\eval\mouth\report.txt
  Saved confusion matrix -> runs\eval\mouth\confusion_matrix.png
  Saved ROC curve        -> runs\eval\mouth\roc_curve.png
